[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20cleaning/practice/regex_worksheet.ipynb)

# Practice · regular expressions

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Ten questions across both regex lectures: classes, quantifiers and anchors from Part 1, then
lookaround, named groups, backreferences and the flags from Part 2. Each question names a variable.
Put your result in that variable and run the cell. The worked answer sits under **Answer**. Click it
open once you have tried.

**The data.** Two files written for this worksheet.

- `lab_notes.csv`: 40 lab reports whose `note` column is free text. Dates in three formats, amounts
  in two currencies, referral codes, phone numbers in two shapes, labelled measurements, and a few
  notes with a word typed twice.
- `access_log.txt`: 60 lines of Apache Common Log Format.


In [ ]:
import re

import numpy as np
import pandas as pd
import requests

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20cleaning/practice/data/"
notes = pd.read_csv(URL + "lab_notes.csv")
log = requests.get(URL + "access_log.txt").text

print(notes.shape, len(log.splitlines()), "log lines")
notes["note"].iloc[0]

### Q1  Referral codes by state

Every note carries one referral code shaped `REF-TN-4099`. Capture the two-letter state
code and count the codes per state.

Answer variable `q1`: a Series indexed by state code.

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
q1 = (notes["note"].str.extract(r"REF-([A-Z]{2})-\d{4}")[0]
        .rename("state").value_counts())
q1
```

```
state
TN    7
KL    7
KA    7
AP    7
MH    6
GJ    6
Name: count, dtype: int64
```

</details>

### Q2  Rupee amounts and dollar amounts

Amounts appear as `₹900` and as `$135`. Total each currency separately, identifying it by
what sits in front of the digits rather than by splitting the string. Strip the thousands comma
before adding.

Answer variable `q2`: a tuple `(rupees, dollars)`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
def total(pattern):
    return sum(int(a.replace(",", ""))
               for n in notes["note"] for a in re.findall(pattern, n))


q2 = (total(r"(?<=₹)([\d,]+)"), total(r"(?<=\$)([\d,]+)"))
q2
```

```
(44200, 579)
```

</details>

### Q3  Why that lookbehind has to be fixed width

Notes could have written the rupee amount as `Rs.900` too, so try to compile
`(?<=₹|Rs\.)[\d,]+`. Report the exception's class name and its message up to the position it
quotes.

Answer variable `q3`: a tuple `(class_name, message)`.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
try:
    re.compile(r"(?<=₹|Rs\.)[\d,]+")
    q3 = None
except Exception as e:
    q3 = (type(e).__name__, str(e).split(" at position")[0])
q3
```

```
('error', 'look-behind requires fixed-width pattern')
```

</details>

### Q4  Labelled measurements

Some notes carry `HbA1c: 7.4%`. Pull every value out by the label in front of it, without
letting the label into the match. Report how many there are and their mean to 2 decimals.

Answer variable `q4`: a tuple `(count, mean)`.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
a1c = [float(v) for n in notes["note"] for v in re.findall(r"(?<=HbA1c: )\d+\.\d", n)]

q4 = (len(a1c), round(float(np.mean(a1c)), 2))
q4
```

```
(14, 7.78)
```

</details>

### Q5  One log line into a record

Parse the first line of `access_log.txt` with named groups for `ip`, `user`, `ts`, `method`,
`path`, `proto`, `status`, `size`, `referer` and `agent`. The second field is the identd, which is
always `-` and is not wanted.

Answer variable `q5`: a dict of 10 strings.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
LINE = re.compile(
    r'(?P<ip>\S+) \S+ (?P<user>\S+) \[(?P<ts>[^\]]+)\] '
    r'"(?P<method>[A-Z]+) (?P<path>\S+) (?P<proto>[^"]+)" '
    r'(?P<status>\d{3}) (?P<size>\d+) "(?P<referer>[^"]*)" "(?P<agent>[^"]*)"')

q5 = LINE.match(log.splitlines()[0]).groupdict()
q5
```

```
{'ip': '132.88.107.108', 'user': 'user0', 'ts': '28/Jun/2026:05:45:24 +0530', 'method': 'GET', 'path': '/', 'proto': 'HTTP/1.1', 'status': '200', 'size': '32170', 'referer': '-', 'agent': 'Mozilla/5.0'}
```

</details>

### Q6  The whole log under re.VERBOSE

Write that pattern again under `re.VERBOSE`, one field per line with a comment, and
`finditer` it over the whole file into a DataFrame. Report the frame's shape, the number of 404s and
the number of requests from GPTBot. In verbose mode unescaped whitespace is discarded, so a literal
space has to be written `[ ]`.

Answer variable `q6`: a tuple `(shape, n_404, n_gptbot)`.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
VERBOSE = re.compile(r"""
    (?P<ip>\S+)[ ]\S+[ ](?P<user>\S+)[ ]      # host, identd, user
    \[(?P<ts>[^\]]+)\][ ]                     # timestamp, in brackets
    "(?P<method>[A-Z]+)[ ](?P<path>\S+)[ ](?P<proto>[^"]+)"[ ]
    (?P<status>\d{3})[ ](?P<size>\d+)[ ]       # status and bytes sent
    "(?P<referer>[^"]*)"[ ]"(?P<agent>[^"]*)"   # referer and user agent
""", re.VERBOSE)

hits = pd.DataFrame(m.groupdict() for m in VERBOSE.finditer(log))
q6 = (hits.shape, int((hits["status"] == "404").sum()),
      int(hits["agent"].str.startswith("GPTBot").sum()))
q6
```

```
((60, 10), 7, 15)
```

</details>

### Q7  Doubled words

A few notes contain a word typed twice in a row. Find them with a backreference, which is
the one construct that compares text against text the pattern already matched. Report the repeated
words, sorted, with no duplicates.

Answer variable `q7`: a sorted list of strings.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
q7 = sorted({m.group(1) for n in notes["note"]
             for m in re.finditer(r"\b(\w+)\s+\1\b", n)})
q7
```

```
['on', 'patient', 'report', 'the', 'was']
```

</details>

### Q8  Normalising the dates with re.sub

`Reviewed on` is followed by `2026-02-18`, `23/01/2026` or `10-Jan-2026`. Rewrite all three
to ISO with a single `re.sub` whose replacement is a function. Then report the earliest date, the
latest, and how many notes ended up carrying one.

Answer variable `q8`: a tuple `(earliest, latest, count)`.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
MON = {m: i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1)}
DATE = re.compile(r"(\d{4})-(\d{2})-(\d{2})"
                  r"|(\d{2})/(\d{2})/(\d{4})"
                  r"|(\d{1,2})-([A-Z][a-z]{2})-(\d{4})")


def iso(m):
    if m.group(1):
        return f"{m[1]}-{m[2]}-{m[3]}"
    if m.group(4):
        return f"{m[6]}-{m[5]}-{m[4]}"
    return f"{m[9]}-{MON[m[8]]:02d}-{int(m[7]):02d}"


fixed = notes["note"].map(lambda n: DATE.sub(iso, n))
seen = fixed.str.extract(r"Reviewed on (\d{4}-\d{2}-\d{2})")[0]

q8 = (seen.min(), seen.max(), int(seen.notna().sum()))
q8
```

```
('2026-01-09', '2026-06-24', 40)
```

</details>

### Q9  Redacting the phone numbers

Mask every phone number in report `LAB-2026-001`, keeping the last four digits and turning
every earlier digit into `X`. The two shapes in the file are `+91 98765 43210` and `9745409135`.
Leave the rest of the note alone, including the amount and the measurement.

Answer variable `q9`: a `str`.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
def mask(m):
    d = re.sub(r"\D", "", m.group(0))
    return "X" * (len(d) - 4) + d[-4:]


q9 = re.sub(r"(?:\+91 )?\d{5}[ -]?\d{5}|\b\d{10}\b", mask,
            notes.loc[notes.report_id == "LAB-2026-001", "note"].iloc[0])
q9
```

```
'[urgent] Reviewed on 2026-02-18 by Dr Iyer. Referral REF-TN-4099. Fasting sample collected at the camp. HbA1c: 8.1%. Consult charged ₹900. Contact XXXXXX9135. [followup]'
```

</details>

### Q10  Greedy against lazy

The first note opens with `[urgent]` and closes with `[followup]`. Match `\[.*\]` against it,
then `\[.*?\]`, and report what each one returns.

Answer variable `q10`: a tuple of two strings.

In [ ]:
q10 = ...   # your answer
q10

<details>
<summary><b>Answer</b></summary>

```python
note1 = notes["note"].iloc[0]

q10 = (re.search(r"\[.*\]", note1).group(), re.search(r"\[.*?\]", note1).group())
q10
```

```
('[urgent] Reviewed on 2026-02-18 by Dr Iyer. Referral REF-TN-4099. Fasting sample collected at the camp. HbA1c: 8.1%. Consult charged ₹900. Contact 9745409135. [followup]', '[urgent]')
```

</details>